In [1]:
print("hello")

hello


In [12]:
from dotenv import load_dotenv
import os
load_dotenv("../.env")

for k in ["ANTHROPIC_API_KEY", "SERPAPI_API_KEY", "PINECONE_API_KEY","PINECONE_INDEX_NAME","PINECONE_NAMESPACE"]:
    print(k, "->", bool(os.getenv(k)))


ANTHROPIC_API_KEY -> True
SERPAPI_API_KEY -> True
PINECONE_API_KEY -> True
PINECONE_INDEX_NAME -> True
PINECONE_NAMESPACE -> True


In [6]:
import anthropic
claude = anthropic.Anthropic()

r = claude.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=200,
    messages=[{"role": "user", "content": "Explain an AI agent in 2 sentences."}],
)
print(r.content[0].text)
print(r.usage)


An AI agent is a software system that perceives its environment, makes decisions, and takes actions on its own to achieve a specific goal. Unlike a basic chatbot that only responds to prompts, an agent can plan multi-step tasks, use tools (like search engines or code), and adapt based on the results it gets.
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=24, output_tokens=100, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard')


In [13]:
import truststore
truststore.inject_into_ssl()

import os, requests
from dotenv import load_dotenv
load_dotenv("../.env")

def web_search(query, num=5):
    r = requests.get(
        "https://serpapi.com/search",
        params={
            "engine": "google",
            "q": query,
            "num": num,
            "api_key": os.getenv("SERPAPI_API_KEY"),
        },
        timeout=30,
    )
    print("Status:", r.status_code)   # keep this while debugging
    r.raise_for_status()
    return r.json().get("organic_results", [])

for item in web_search("what is agentic AI"):
    print(item["title"], "|", item["link"])
    print(item.get("snippet", ""), "\n")


Status: 200
What is Agentic AI? | https://www.ibm.com/think/topics/agentic-ai
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of ai agents—machine learning ... 

Agentic AI, explained | https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained
AI agents or agentic AI, a new breed of AI systems that are semi- or fully autonomous and thus able to perceive, reason, and act on their own. 

What is agentic AI? Definition and differentiators | https://cloud.google.com/discover/what-is-agentic-ai
Agentic AI is an advanced form of artificial intelligence focused on autonomous decision-making and action. Unlike traditional AI, which ... 

What is Agentic AI? - AWS - Amazon.com | https://aws.amazon.com/what-is/agentic-ai/
Agentic AI is an autonomous AI system that can act independently to achieve pre-determined goals. agentic AI is proactive and can perform complex tasks without ... 

What is Agentic AI? | https:/

In [ ]:
import truststore
truststore.inject_into_ssl()

from dotenv import load_dotenv
from pinecone import Pinecone
load_dotenv("../.env")

pc = Pinecone(api_key="")
print(pc.list_indexes().names())


['enterprise-ai-agents']


In [6]:
import truststore
truststore.inject_into_ssl()
from langchain_community.document_loaders import WebBaseLoader

def load_web_pages(urls, user_agent="Mozilla/5.0 Agentic-RAG-Demo"):
    """
    Load one or more web pages and return a list of LangChain Documents.

    Args:
        urls: a single URL (str) or a list of URLs
        user_agent: User-Agent header sent with the request

    Returns:
        list of Document objects (one per URL)
    """
    # Allow passing a single string
    if isinstance(urls, str):
        urls = [urls]

    loader = WebBaseLoader(
        web_paths=urls,
        requests_kwargs={"headers": {"User-Agent": user_agent}},
    )

    try:
        docs = loader.load()
    except Exception as e:
        print(f"Error loading pages: {e}")
        return []

    return docs

docs = load_web_pages("https://docs.langchain.com/oss/python/langgraph/agentic-rag")
print(docs)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain', 'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en'}, page_content='\n\n\n\n\n\nBuild a custom RAG agent with LangGraph - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep Re

In [ ]:
import truststore
truststore.inject_into_ssl()
import re
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf(
    file_path,
    extract_images=False,
    clean=True,
    min_chars=20,
    verbose=True,
):
    """
    Load a PDF using PyMuPDFLoader and return a list of LangChain Documents
    (one per page).

    Args:
        file_path (str | Path): path to the PDF file.
        extract_images (bool): run OCR on images inside the PDF
                               (needs: pip install rapidocr-onnxruntime).
        clean (bool): normalize whitespace in the extracted text.
        min_chars (int): pages with fewer characters than this are dropped.
        verbose (bool): print a summary of what was loaded.

    Returns:
        list[Document]: Documents with cleaned text and enriched metadata.
    """
    # 1. Validate the file
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Not a PDF file: {path.name}")

    # 2. Load with PyMuPDFLoader
    try:
        loader = PyMuPDFLoader(str(path), extract_images=extract_images)
        docs = loader.load()
    except Exception as e:
        raise RuntimeError(f"Failed to load '{path.name}': {e}")

    total_pages = len(docs)
    cleaned_docs = []
    skipped_pages = []

    # 3. Clean, filter and enrich each page
    for doc in docs:
        text = doc.page_content

        if clean:
            text = re.sub(r"[ \t]+", " ", text)       # collapse spaces and tabs
            text = re.sub(r"\n{3,}", "\n\n", text)    # max one blank line
            text = text.strip()

        page_index = doc.metadata.get("page")

        # Skip blank or image-only pages
        if len(text) < min_chars:
            skipped_pages.append(page_index)
            continue

        doc.page_content = text
        doc.metadata["file_name"] = path.name
        if isinstance(page_index, int):
            doc.metadata["page_number"] = page_index + 1   # starts at 1

        cleaned_docs.append(doc)

    # 4. Summary and warnings
    if verbose:
        print(f"File          : {path.name}")
        print(f"Total pages   : {total_pages}")
        print(f"Pages kept    : {len(cleaned_docs)}")
        print(f"Pages skipped : {len(skipped_pages)}")

    if not cleaned_docs:
        print(
            "WARNING: no text was extracted. The PDF may be scanned (image-only). "
            "Try extract_images=True or an OCR-based approach."
        )

    return cleaned_docs

docs = load_pdf("C:\\Users\\prince.mahato\\Downloads\\301WhiteBlossomCirExclusiveRighttoSell.pdf")

print(len(docs))                    # number of pages with text
print(docs[0].page_content[:300])   # first 300 characters of page 1
print(docs[0].metadata)

RuntimeError: Failed to load '301WhiteBlossomCirExclusiveRighttoSell.pdf' with pypdf: `pypdf` package not found, please install it with `pip install pypdf`

RuntimeError: Failed to load '301WhiteBlossomCirExclusiveRighttoSell.pdf' with pypdf: `pypdf` package not found, please install it with `pip install pypdf`